# Prajna Training Only
**Loads existing data → trains E2B with full CRN → saves checkpoints**

Upload `teacher_data.json` via Colab sidebar (📁 icon → Upload).
**Required:** Runtime → Change runtime type → T4 GPU

**Expected time:** ~25 min for 3000 samples × 5 epochs

In [ ]:
# Cell 1: Install & Setup
!pip install -q torch transformers accelerate bitsandbytes einops

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os, json, time

from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR='/content/drive/MyDrive/prajna/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'Checkpoints will save to: {CKPT_DIR}')
print('Upload teacher_data.json via Colab sidebar.')


In [ ]:
# Cell 2: Full CRN Components
import torch.nn as nn
import torch.nn.functional as F

class ResonanceAttention(nn.Module):
    def __init__(self, d_model, num_heads=4, num_frequencies=16, top_k=4):
        super().__init__()
        self.num_heads = num_heads; self.num_frequencies = num_frequencies
        self.top_k = top_k; self.head_dim = d_model // num_heads
        self.freq_q = nn.Linear(d_model, num_heads * num_frequencies, bias=False)
        self.freq_k = nn.Linear(d_model, num_heads * num_frequencies, bias=False)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, x):
        B,T,D = x.shape; device = x.device
        q = self.freq_q(x).view(B,T,self.num_heads,self.num_frequencies)
        k = self.freq_k(x).view(B,T,self.num_heads,self.num_frequencies)
        freq_scores = F.softmax(q, dim=-1)
        top_freq_vals, top_freq_idx = freq_scores.topk(min(self.top_k,self.num_frequencies), dim=-1)
        top_freq_vals = top_freq_vals / (top_freq_vals.sum(dim=-1,keepdim=True)+1e-8)
        v = self.v_proj(x).view(B,T,self.num_heads,self.head_dim)
        out = torch.zeros_like(v)
        for f_idx in range(self.num_frequencies):
            mask = (top_freq_idx == f_idx).any(dim=-1)
            if mask.sum() == 0: continue
            freq_weight = torch.zeros(B,T,self.num_heads,device=device)
            for k_idx in range(self.top_k):
                match = (top_freq_idx[:,:,:,k_idx]==f_idx)
                freq_weight += match.float() * top_freq_vals[:,:,:,k_idx]
            q_f = q[:,:,:,f_idx]; k_f = k[:,:,:,f_idx]
            attn_scores = torch.einsum('bih,bjh->bhij', q_f, k_f) / (self.head_dim**0.5)
            attn_mask = mask.unsqueeze(2)*mask.unsqueeze(1)
            attn_scores = attn_scores.masked_fill(~attn_mask.permute(0,3,1,2).bool(), float('-inf'))
            attn_weights = F.softmax(attn_scores, dim=-1).nan_to_num(0.0)
            out += freq_weight.unsqueeze(-1) * torch.einsum('bhij,bjhd->bihd', attn_weights, v)
        return self.out_proj(out.reshape(B,T,D))

class EpisodicMemory:
    def __init__(self, d_model, mem_size=512, mem_dim=64, device='cpu'):
        self.mem_size = mem_size; self.mem_dim = mem_dim; self.d_model = d_model; self.device = device
        self.memory = torch.zeros(mem_size,mem_dim,device=device,requires_grad=False)
        self.temporal_positions = torch.zeros(mem_size,device=device,requires_grad=False)
        self.write_ptr = 0; self.step_count = 0
        self.compress = nn.Linear(d_model,mem_dim)
        self.decompress = nn.Linear(mem_dim,d_model)
        self.read_gate = nn.Linear(d_model,mem_dim)
        self.write_gate = nn.Linear(d_model,1)
        self.relevance_gate = nn.Linear(d_model+mem_dim,1)
    def get_parameters(self):
        return list(self.compress.parameters())+list(self.decompress.parameters())+list(self.read_gate.parameters())+list(self.write_gate.parameters())+list(self.relevance_gate.parameters())
    def read(self, query, top_k=8):
        if query.dim()==1: query=query.unsqueeze(0)
        B=query.shape[0]
        q_compressed=self.read_gate(query)
        mem_expanded=self.memory.unsqueeze(0).expand(B,-1,-1)
        q_norm=F.normalize(q_compressed,dim=-1); mem_norm=F.normalize(mem_expanded,dim=-1)
        sims=torch.bmm(q_norm.unsqueeze(1),mem_norm.transpose(1,2)).squeeze(1)
        recency=self.temporal_positions/(self.temporal_positions.max()+1)
        sims=sims+0.1*recency.unsqueeze(0)
        top_k=min(top_k,self.mem_size)
        top_vals,top_idx=sims.topk(top_k,dim=-1)
        attn_weights=F.softmax(top_vals,dim=-1)
        retrieved=torch.gather(mem_expanded,1,top_idx.unsqueeze(-1).expand(-1,-1,self.mem_dim))
        retrieved=(retrieved*attn_weights.unsqueeze(-1)).sum(dim=1)
        return self.decompress(retrieved), attn_weights
    def write(self, content, force=False):
        gate_value=torch.sigmoid(self.write_gate(content.unsqueeze(0))).item()
        if gate_value<0.5 and not force: return False
        compressed=self.compress(content.detach())
        if self.write_ptr<self.mem_size:
            slot=self.write_ptr; self.write_ptr+=1
        else: slot=self.temporal_positions.argmin().item()
        write_weight=min(gate_value,0.9)
        self.memory[slot]=(write_weight*compressed+(1-write_weight)*self.memory[slot].clone()).detach()
        self.step_count+=1; self.temporal_positions[slot]=self.step_count
        return True
    def save(self, path):
        state={'memory':self.memory.detach().cpu().float().numpy().tolist(),'temporal_positions':self.temporal_positions.detach().cpu().float().numpy().tolist(),'write_ptr':self.write_ptr,'step_count':self.step_count}
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.',exist_ok=True)
        with open(path,'w') as f: json.dump(state,f)
    def load(self, path):
        with open(path) as f: state=json.load(f)
        self.memory=torch.tensor(state['memory'],dtype=torch.float32,device=self.device)
        self.temporal_positions=torch.tensor(state['temporal_positions'],dtype=torch.float32,device=self.device)
        self.write_ptr=state['write_ptr']; self.step_count=state['step_count']
    def get_stats(self):
        return {'used_slots':(self.temporal_positions>0).sum().item(),'total_slots':self.mem_size,'write_ptr':self.write_ptr}

class ReflectiveLoop(nn.Module):
    def __init__(self, d_model, num_corrections=16):
        super().__init__()
        self.num_corrections=num_corrections; self.d_model=d_model
        self.critic=nn.Sequential(nn.Linear(d_model,d_model//4),nn.GELU(),nn.Linear(d_model//4,num_corrections+1))
        self.correction_directions=nn.Parameter(torch.randn(num_corrections,d_model)*0.01)
        self.thresholds=nn.Parameter(torch.ones(num_corrections)*0.5)
        self.confidence_scale=nn.Parameter(torch.tensor(0.1))
    def forward(self, hidden_state, return_correction_id=False):
        pooled=hidden_state.mean(dim=1) if hidden_state.dim()==3 else hidden_state
        scores=self.critic(pooled)
        no_correction_score=scores[:,-1]; correction_scores=scores[:,:-1]
        best_score,best_idx=correction_scores.max(dim=-1)
        apply_correction=best_score>(no_correction_score+0.2)
        corrected_state=hidden_state.clone(); correction_id=-1
        if apply_correction.any():
            for b in range(hidden_state.shape[0]):
                if apply_correction[b]:
                    correction=self.correction_directions[best_idx[b]]
                    confidence=torch.sigmoid(best_score[b]-self.thresholds[best_idx[b]])
                    scale=torch.abs(self.confidence_scale)
                    corrected_state[b]=hidden_state[b]+scale*confidence*correction
                    correction_id=best_idx[b].item()
        return (corrected_state, correction_id) if return_correction_id else corrected_state
    def compute_loss(self, hidden_state, is_error, correct_direction=None):
        pooled=hidden_state.mean(dim=1) if hidden_state.dim()==3 else hidden_state
        scores=self.critic(pooled)
        targets=torch.where(is_error,correct_direction,torch.full_like(correct_direction,self.num_corrections))
        return F.cross_entropy(scores,targets)
    def get_correction_stats(self):
        return {'num_corrections':self.num_corrections,'thresholds_mean':self.thresholds.mean().item(),'confidence_scale':torch.abs(self.confidence_scale).item()}

class SkillComposer(nn.Module):
    def __init__(self, d_model, num_skills=64, skill_rank=8, top_k=4):
        super().__init__()
        self.num_skills=num_skills; self.skill_rank=skill_rank; self.top_k=top_k; self.d_model=d_model
        self.skill_u=nn.Parameter(torch.randn(num_skills,d_model,skill_rank)*0.01)
        self.skill_v=nn.Parameter(torch.randn(num_skills,skill_rank,d_model)*0.01)
        self.router=nn.Sequential(nn.Linear(d_model,d_model//4),nn.GELU(),nn.Linear(d_model//4,num_skills))
        self.skill_scale=nn.Parameter(torch.ones(num_skills)*0.01)
    def forward(self, x, return_skill_info=False):
        B,T,D=x.shape
        skill_logits=self.router(x.mean(dim=1))
        skill_weights=F.softmax(skill_logits,dim=-1)
        if self.training:
            self._load_balance_loss=(skill_weights.mean(dim=0).var()*10.0)
        else: self._load_balance_loss=torch.tensor(0.0)
        top_k=min(self.top_k,self.num_skills)
        top_weights,top_indices=skill_weights.topk(top_k,dim=-1)
        top_weights=top_weights/(top_weights.sum(dim=-1,keepdim=True)+1e-8)
        perturbation=torch.zeros_like(x)
        for k in range(self.top_k):
            u=self.skill_u[top_indices[:,k]]; v=self.skill_v[top_indices[:,k]]
            scale=torch.abs(self.skill_scale[top_indices[:,k]])
            x_v=torch.bmm(x,v.transpose(1,2))
            perturbation+=top_weights[:,k].unsqueeze(1).unsqueeze(-1)*scale.unsqueeze(1).unsqueeze(-1)*torch.bmm(x_v,u.transpose(1,2))
        return x+perturbation

class CRNMemoryLayer(nn.Module):
    def __init__(self, d_model, mem_size=512, mem_dim=128, device='cpu'):
        super().__init__(); self.d_model=d_model; self.device=device
        self.memory=EpisodicMemory(d_model,mem_size,mem_dim,device)
        for mod in [self.memory.compress,self.memory.decompress,self.memory.read_gate,self.memory.write_gate,self.memory.relevance_gate]:
            mod.to(device, dtype=torch.bfloat16)
        self.memory.memory=self.memory.memory.to(device, dtype=torch.bfloat16)
        self.memory.temporal_positions=self.memory.temporal_positions.to(device, dtype=torch.bfloat16)
        self.read_gate=nn.Linear(d_model,d_model,dtype=torch.bfloat16).to(device)
        self.write_gate=nn.Linear(d_model,1,dtype=torch.bfloat16).to(device)
        self.blend=nn.Parameter(torch.tensor(0.1,dtype=torch.bfloat16,device=device))
    def read(self,h):
        if self.memory.temporal_positions.sum()==0: return h
        retrieved,_=self.memory.read(self.read_gate(h.mean(dim=1)),top_k=8)
        return h+torch.sigmoid(self.blend)*retrieved.unsqueeze(1)
    def write(self,h):
        if self.training: self.memory.write(h[:,-1,:].mean(dim=0),force=False)
    def get_parameters(self):
        return self.memory.get_parameters()+list(self.read_gate.parameters())+list(self.write_gate.parameters())+[self.blend]
    def save(self,p): self.memory.save(p)
    def load(self,p): self.memory.load(p)

print('CRN components loaded')

In [ ]:
# Cell 3: Student with Full CRN
class PrajnaStudent(nn.Module):
    def __init__(self, device='cuda'):
        super().__init__(); self.device=device
        print('Loading E2B student...')
        self.tok=AutoTokenizer.from_pretrained('google/gemma-4-E2B')
        self.model=AutoModelForCausalLM.from_pretrained('google/gemma-4-E2B',torch_dtype=torch.bfloat16)
        self.model.to('cuda')
        self.mem=CRNMemoryLayer(d_model=1536,mem_size=512,mem_dim=128,device=device)
        self.reflection=ReflectiveLoop(d_model=1536,num_corrections=16).to(device, dtype=torch.bfloat16)
        self.skills=SkillComposer(d_model=1536,num_skills=64,skill_rank=8,top_k=4).to(device, dtype=torch.bfloat16)
        self.resonance=ResonanceAttention(d_model=1536,num_heads=4,num_frequencies=16,top_k=4).to(device, dtype=torch.bfloat16)
        self.vocab=262144
        for p in self.model.parameters(): p.requires_grad=False
        self._hooks=[]
        layers=self.model.model.language_model.layers; mid=len(layers)//2
        self._hooks.append(layers[mid].register_forward_pre_hook(lambda m,i:(self.mem.read(i[0].to(self.device,dtype=torch.bfloat16)).to(i[0].device),)+i[1:]))
        self._hooks.append(layers[-1].register_forward_hook(lambda m,i,o:(self.mem.write(o[0] if isinstance(o,tuple) else o),o)[1]))
        for i,l in enumerate(layers):
            if i%4==0:
                self._hooks.append(l.register_forward_hook(lambda m,i,o:((self.reflection((o[0] if isinstance(o,tuple) else o).to(self.device,dtype=torch.bfloat16)).to(o.device),)+o[1:]) if isinstance(o,tuple) else self.reflection(o.to(self.device,dtype=torch.bfloat16)).to(o.device)))
        for i,l in enumerate(layers):
            if i%4==0:
                self._hooks.append(l.register_forward_hook(lambda m,inp,o:((self.skills((o[0] if isinstance(o,tuple) else o).to(self.device,dtype=torch.bfloat16)).to(o.device),)+o[1:]) if isinstance(o,tuple) else self.skills(o.to(self.device,dtype=torch.bfloat16)).to(o.device)))
        for i,l in enumerate(layers):
            if i%4==0 and i<mid:
                self._hooks.append(l.register_forward_hook(lambda m,inp,o:((self.resonance((o[0] if isinstance(o,tuple) else o).to(self.device,dtype=torch.bfloat16)).to(o.device),)+o[1:]) if isinstance(o,tuple) else self.resonance(o.to(self.device,dtype=torch.bfloat16)).to(o.device)))
        params=self.get_params()
        print(f'CRN: {sum(p.numel() for p in params):,} params, {len(self._hooks)} hooks')
    def forward(self, input_ids, labels=None):
        out=self.model(input_ids=input_ids)
        loss=None
        if labels is not None:
            ce_loss=F.cross_entropy(out.logits[:,:-1].reshape(-1,self.vocab),labels[:,1:].reshape(-1),ignore_index=-100)
            with torch.no_grad():
                preds=out.logits[:,:-1].argmax(dim=-1)
                is_error=(preds!=labels[:,1:])&(labels[:,1:]!=-100)
            contrastive_loss=torch.tensor(0.0,device=self.device)
            if is_error.any():
                if not hasattr(self,'_proj'):
                    self._proj=nn.Linear(self.vocab,1536,dtype=torch.bfloat16).to(self.device)
                proj_hidden=self._proj(out.logits.mean(dim=1).unsqueeze(1))
                correct_dir=torch.randint(0,16,(is_error.shape[0],),device=self.device)
                contrastive_loss=self.reflection.compute_loss(proj_hidden,is_error.any(dim=-1),correct_dir)
            loss=ce_loss+0.1*contrastive_loss
        return {'loss':loss,'ce_loss':ce_loss.item() if labels is not None else 0}
    def get_params(self):
        p=self.mem.get_parameters()+list(self.reflection.parameters())+list(self.skills.parameters())+list(self.resonance.parameters())
        if hasattr(self,'_proj'): p+=list(self._proj.parameters())
        return p
    def save_memory(self,p): self.mem.save(p)
    def load_memory(self,p): self.mem.load(p)
    def cleanup(self):
        for h in self._hooks: h.remove(); self._hooks.clear()

print('Student class defined')

In [ ]:
# Cell 4: Load Data
data_file='/content/teacher_data.json'
samples = []
if not os.path.exists(data_file):
    print(f"ERROR: Data file not found at {data_file}")
    print('Upload teacher_data.json via Colab sidebar')
else:
    with open(data_file) as f: samples=json.load(f)
    print(f'Data loaded: {len(samples)} samples')


In [ ]:
# Cell 5: Create Student
student=PrajnaStudent(device='cuda')
opt=torch.optim.AdamW(student.get_params(),lr=2e-4)
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'Memory: {student.mem.memory.get_stats()}')
print(f'Reflection: {student.reflection.get_correction_stats()}')

In [ ]:
# Cell 6: Training
from torch.utils.data import Dataset, DataLoader

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.samples=data; self.tok=tokenizer; self.ml=max_length
    def __len__(self): return max(len(self.samples),1)
    def __getitem__(self,i):
        if not self.samples:
            d=torch.zeros(self.ml,dtype=torch.long); return {'input_ids':d,'labels':d.clone()}
        s=self.samples[i%len(self.samples)]
        text=f"{s.get('prompt','')}\n\n{s.get('response','')}"
        enc=self.tok(text,truncation=True,max_length=self.ml,padding='max_length',return_tensors='pt')
        ids=enc['input_ids'].squeeze(); labels=ids.clone()
        labels[enc['attention_mask'].squeeze()==0]=-100
        return {'input_ids':ids,'labels':labels}

dataset=SimpleDataset(samples,student.tok) if samples else None
if not samples or not dataset or len(dataset)==0:
    raise RuntimeError('No data loaded! Run Cell 4 first and make sure teacher_data.json is uploaded.')
loader=DataLoader(dataset,batch_size=1,shuffle=True)

print('='*60); print('TRAINING STARTED'); print('='*60)
losses=[]; t_start=time.time()
ckpt_dir=CKPT_DIR

for epoch in range(5):
    print(f'Epoch {epoch+1}/5')
    for i,batch in enumerate(loader):
        input_ids=batch['input_ids'].to('cuda')
        labels=batch['labels'].to('cuda')
        student.train()
        out=student(input_ids,labels)
        loss=out['loss']
        if torch.isnan(loss): continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.get_params(),1.0)
        opt.step(); opt.zero_grad()
        losses.append(loss.item())
        if len(losses)%10==0:
            avg=sum(losses[-10:])/10
            mem=student.mem.memory.get_stats()
            print(f'  Step {len(losses):5d} | Loss: {loss.item():.4f} | Avg: {avg:.4f} | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')
        if len(losses)%500==0:
            torch.save({'step':len(losses),'crn':{k:v for k,v in student.state_dict().items() if not k.startswith('model')},'loss':sum(losses[-50:])/len(losses[-50:]),'memory_stats':student.mem.memory.get_stats()},f'{ckpt_dir}/ckpt_{len(losses)}.pt')
            student.save_memory(f'{ckpt_dir}/memory_{len(losses)}.json')
            print(f'  Checkpoint + memory saved!')

final_loss=sum(losses[-50:])/len(losses[-50:]) if losses else 0
torch.save({'step':len(losses),'crn':{k:v for k,v in student.state_dict().items() if not k.startswith('model')},'loss':final_loss,'memory_stats':student.mem.memory.get_stats(),'correction_stats':student.reflection.get_correction_stats()},f'{ckpt_dir}/best.pt')
student.save_memory(f'{ckpt_dir}/memory_best.json')
elapsed=(time.time()-t_start)/60
print(f'Steps: {len(losses)} | Final loss: {final_loss:.4f} | Time: {elapsed:.1f} min')
print(f'Memory stats: {student.mem.memory.get_stats()}')
print('Done!')

In [ ]:
# Cell 7: Verify Save
import glob
ckpt=torch.load(f'{ckpt_dir}/best.pt',weights_only=False)
print(f'Steps: {ckpt["step"]} | Loss: {ckpt["loss"]:.4f}')
print(f'CRN keys: {list(ckpt["crn"].keys())[:5]}')
print(f'Memory stats: {ckpt.get("memory_stats",{})}')
print(f'Checkpoints:')
for f in sorted(glob.glob(f'{ckpt_dir}/*')):
    print(f'  {os.path.basename(f)}')
student.cleanup()